# Immortalite Zero — Self-Play Training (Kaggle)

Trains on a free Kaggle GPU. Checkpoints write to `/kaggle/working/results` every iteration.

**Persistence:** Kaggle has no Drive. Before you stop the session, download `/kaggle/working/results` or create/update a private Dataset from it, then **Add Input** that Dataset on the next run so cell 3 can resume from `latest.pt`.

Optional: add a Syzygy Dataset (145 `.rtbw` files). Cell 5 uses it if present, otherwise downloads once into `/kaggle/working/syzygy345`.

**Run cells 1→8 top to bottom.** Edit only the `TRAIN` dict in cell 6. Same recipe as Colab (`selfplay_workers=2`).

## Current recipe (cell 6)

Resume from **`latest.pt`** (iter **261+** recipe). See changelog.

| Setting | Value |
|---------|-------|
| Self-play | **128 games/iter**, **150 MCTS sims**, **concurrency 128**, **2 workers** |
| Move temperature | **T=4.0** for first **10** plies (sampling only; targets untempered) |
| Training | 800 steps/iter, LR **2.5e-4 flat**, `value_target=root_q` |
| Replay | **200k** buffer/window |
| Draw shaping | **1/3** in self-play; gates use normal WDL |
| Resign | **off** |
| Training span | auto-stops after completing iters **…, 260, 280, …** (multiples of 20) |
| In-loop gate | **off** — run **manual gate (cell 7)** when you want SPRT |

**Manual gate (cell 7):** SPRT cap **128 games**, **100 sims**, masters book (64×2 colors).

Recipe history: [TRAINING_CHANGELOG.md](https://github.com/carlo-wong/immortalite-zero/blob/main/TRAINING_CHANGELOG.md)

Set accelerator to **GPU** first (Notebook → Session options → Accelerator).


In [ ]:
# 1. Get the code. Safe to re-run (always works from /kaggle/working, never nests).
import os
%cd /kaggle/working
if not os.path.exists('/kaggle/working/immortalite-zero'):
    !git clone https://github.com/carlo-wong/immortalite-zero.git
%cd /kaggle/working/immortalite-zero
!git pull --quiet
print('working dir:', os.getcwd())


In [ ]:
# 2. Install dependencies (Kaggle already has a CUDA torch build).
!pip install -q python-chess numpy tqdm


In [ ]:
# 3. Checkpoint dir + optional Dataset resume.
# Add a private Dataset of your previous /kaggle/working/results (must contain latest.pt).
# Cell auto-detects any /kaggle/input/* (or */results) that has latest.pt and copies into CKPT_DIR.
import os
import shutil
from pathlib import Path

import torch

INPUT_ROOT = Path('/kaggle/input')
CKPT_DIR = '/kaggle/working/results'
os.makedirs(CKPT_DIR, exist_ok=True)


def _checkpoint_iter(path):
    try:
        state = torch.load(path, map_location='cpu')
        if isinstance(state, dict):
            return int(state.get('iteration', -1))
    except Exception:
        pass
    return -1


def _find_results_src():
    if not INPUT_ROOT.is_dir():
        return None
    candidates = []
    for name in sorted(os.listdir(INPUT_ROOT)):
        root = INPUT_ROOT / name
        if not root.is_dir():
            continue
        for candidate in (root, root / 'results'):
            latest = candidate / 'latest.pt'
            if latest.is_file():
                candidates.append(candidate)
    if not candidates:
        return None

    def score(p):
        slug = p.parts[3] if len(p.parts) > 3 else p.name
        latest = p / 'latest.pt'
        return (
            -_checkpoint_iter(latest),  # highest iteration first
            -latest.stat().st_mtime,    # then newest file
            0 if 'immortalite' in slug.lower() else 1,
            0 if 'result' in slug.lower() else 1,
            slug,
        )

    return str(sorted(candidates, key=score)[0])


src = _find_results_src()
if src is None:
    print('fresh start: no input Dataset with latest.pt')
    print('checkpoints ->', CKPT_DIR)
else:
    print('resume source ->', src)
    copied = 0
    for name in os.listdir(src):
        s = os.path.join(src, name)
        d = os.path.join(CKPT_DIR, name)
        if os.path.isfile(s):
            if (not os.path.exists(d)) or os.path.getmtime(s) > os.path.getmtime(d):
                shutil.copy2(s, d)
                copied += 1
        elif os.path.isdir(s) and name.startswith('samples'):
            if not os.path.exists(d):
                shutil.copytree(s, d)
                copied += 1
    print(f'copied/updated {copied} items into {CKPT_DIR}')

if os.path.exists(os.path.join(CKPT_DIR, 'latest.pt')):
    print('resume ready: latest.pt found — cell 6 will continue from saved iteration')
else:
    print('fresh start: no latest.pt — cell 6 will begin at iter 0')
print('checkpoints ->', CKPT_DIR)


In [ ]:
# 4. Confirm GPU and set CLI preset. Cell 6 TRAIN dict overrides --gpu defaults.
import os
import torch

HAS_CUDA = torch.cuda.is_available()
CUDA_DEVICE = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'
N_GPU = torch.cuda.device_count() if HAS_CUDA else 0
CPU_COUNT = os.cpu_count() or 1
TRAIN_PRESET = '--device cuda --gpu' if HAS_CUDA else '--device cpu --light'

print('CUDA available:', HAS_CUDA)
print('device:', CUDA_DEVICE, f'(count={N_GPU})')
print('cpu_count:', CPU_COUNT)
print('train preset:', TRAIN_PRESET)
print('tip: keep selfplay_workers=2 unless you confirm >=4 usable vCPUs')


In [ ]:
# 5. Syzygy 3-4-5 WDL — prefer a Kaggle Dataset with .rtbw files, else download.
import os
import re
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

TB_DIR = '/kaggle/working/syzygy345'
INPUT_ROOT = Path('/kaggle/input')
TB_MIRRORS = [
    'http://tablebase.sesse.net/syzygy/3-4-5/',
]
EXPECTED_RTBW = 145
os.makedirs(TB_DIR, exist_ok=True)


def _find_syzygy_src():
    if not INPUT_ROOT.is_dir():
        return None
    best = None
    best_n = 0
    for name in sorted(os.listdir(INPUT_ROOT)):
        root = INPUT_ROOT / name
        if not root.is_dir():
            continue
        for candidate in (root, root / 'syzygy345'):
            if not candidate.is_dir():
                continue
            n = sum(1 for f in os.listdir(candidate) if f.endswith('.rtbw'))
            if n > best_n:
                best_n = n
                best = candidate
    return str(best) if best is not None and best_n > 0 else None


def _local_rtbw_count():
    return len([f for f in os.listdir(TB_DIR) if f.endswith('.rtbw')])


src = _find_syzygy_src()
if src is not None:
    print(f'Syzygy Dataset -> {src}')
    copied = 0
    for name in os.listdir(src):
        if not name.endswith('.rtbw'):
            continue
        s = os.path.join(src, name)
        d = os.path.join(TB_DIR, name)
        if not os.path.exists(d):
            shutil.copy2(s, d)
            copied += 1
    print(f'copied {copied} .rtbw into {TB_DIR}')

local_count = _local_rtbw_count()
if local_count >= EXPECTED_RTBW:
    print(f'Syzygy ready -> {TB_DIR} ({local_count} files)')
else:
    index_html = None
    TB_BASE_URL = None
    for mirror in TB_MIRRORS:
        try:
            index_html = urllib.request.urlopen(mirror, timeout=30).read().decode('utf-8', errors='ignore')
            TB_BASE_URL = mirror
            break
        except Exception as exc:
            print(f'Failed mirror {mirror}: {exc}')

    if index_html is None or TB_BASE_URL is None:
        raise RuntimeError(
            f'Could not reach Syzygy mirror and Dataset incomplete ({local_count}/{EXPECTED_RTBW}).'
        )

    files = sorted(set(re.findall(r'href="([^"/]+\.rtbw)"', index_html)))
    if not files:
        # Mirror reachable but listing format changed — still OK if Dataset already filled TB_DIR.
        local_count = _local_rtbw_count()
        if local_count >= EXPECTED_RTBW:
            print(f'Syzygy ready from Dataset (mirror listing empty) -> {TB_DIR} ({local_count} files)')
        else:
            raise RuntimeError(f'No .rtbw files found at Syzygy source URL: {TB_BASE_URL}')
    else:
        print(f'Syzygy WDL source -> {TB_BASE_URL}')
        print(f'Expected files: {len(files)}')
        downloaded = 0
        for idx, name in enumerate(files, start=1):
            dst = os.path.join(TB_DIR, name)
            if os.path.exists(dst):
                continue
            src_url = urllib.parse.urljoin(TB_BASE_URL, name)
            urllib.request.urlretrieve(src_url, dst)
            downloaded += 1
            if idx % 25 == 0 or idx == len(files):
                print(f'  progress {idx}/{len(files)}')
        local_count = _local_rtbw_count()
        print(f'downloaded: {downloaded}')
        print(f'Syzygy ready -> {TB_DIR} ({local_count}/{len(files)} files)')
        if local_count < len(files):
            raise RuntimeError('Syzygy setup incomplete — check Dataset upload or network.')


In [ ]:
# 6. Config + train. Edit TRAIN below only — matches colab/train.ipynb / lightning-ai/run_train.py.
#
# Current recipe: 150 MCTS sims (self-play), 128 games/iter, 2 workers, concurrency 128.
# Start with 2 workers on Kaggle; raise to 4 only if cpu_count (cell 4) supports it.
#
# No in-loop gates — run manual gate in cell 7 when you want SPRT.
# Auto-stops after completing an iter that is a multiple of STOP_INTERVAL (160, 180, …).
import os
import torch

if 'TRAIN_PRESET' not in globals():
    raise RuntimeError('Run cell 4 first.')
if 'CKPT_DIR' not in globals():
    raise RuntimeError('Run cell 3 first.')
if 'TB_DIR' not in globals():
    raise RuntimeError('Run cell 5 first.')

STOP_INTERVAL = 20

TRAIN = {
    'sims': 150,
    'gate_sims': 100,   # manual gate cell 7 only
    'games': 128,
    'train_steps': 800,
    'concurrency': 128,
    'selfplay_workers': 2,
    'gate_workers': 2,
    'replay_buffer': 200_000,
    'replay_window': 200_000,
    'draw_penalty': 1 / 3,  # football 3-1-0 shaping still sets search draw_contempt
    'value_target': 'root_q',  # per-ply MCTS root Q labels (option B)
    'gate_games': 128,      # manual gate cell 7 only
    'gate_concurrency': 128,
    'gate_exploration_moves': 0,  # after masters book; no temperature
    'gate_openings': 'masters',  # 64 lines x 2 colors (=128 games)
    'save_every': 10,
    'resume': True,    # on by default — survives session end if you save results Dataset
    'resign': False,
    'lr': 2.5e-4,
    'lr_min': 2.5e-4,
    'lr_total_iters': 10_000,
    'lr_warmup_iters': 0,
    'grad_clip': 10.0,
    # Early-ply move sampling temperature (policy targets stay untempered).
    'move_temperature': 4.0,
    'move_temperature_plies': 10,
}
RESET_OPTIMIZER = False
RESIGN_THRESHOLD = -0.90
RESIGN_PLIES = 3
RESIGN_MIN_MOVES = 20


def _training_span(resume_path, resume, stop_interval):
    start_iter = 0
    if resume and resume_path and os.path.exists(resume_path):
        state = torch.load(resume_path, map_location='cpu')
        start_iter = int(state.get('iteration', -1)) + 1
    if start_iter % stop_interval == 0:
        end_iter = start_iter
    else:
        end_iter = ((start_iter // stop_interval) + 1) * stop_interval
    return start_iter, end_iter, end_iter - start_iter + 1


resume_path = os.path.join(CKPT_DIR, 'latest.pt')
start_iter, end_iter, train_iterations = _training_span(
    resume_path, TRAIN['resume'], STOP_INTERVAL,
)

resume_arg = ''
if TRAIN['resume']:
    if os.path.exists(resume_path):
        resume_arg = f'--resume {resume_path}'
    else:
        print('WARNING: resume=True but no latest.pt — starting at iter 0')

reset_opt_arg = '--reset-optimizer' if RESET_OPTIMIZER else ''

syzygy_arg = f'--syzygy-path "{TB_DIR}"'
resign_arg = ''
if TRAIN['resign']:
    resign_arg = (
        f'--resign-threshold {RESIGN_THRESHOLD} '
        f'--resign-plies {RESIGN_PLIES} '
        f'--resign-min-moves {RESIGN_MIN_MOVES}'
    )

print('TRAIN config:', TRAIN)
print(f'RESET_OPTIMIZER={RESET_OPTIMIZER} preset={TRAIN_PRESET} ckpt_dir={CKPT_DIR}')
print(
    f'training span: iters {start_iter}..{end_iter} '
    f'({train_iterations} iterations), stop_interval={STOP_INTERVAL}'
)

!python -m engine.train --iterations {train_iterations} {TRAIN_PRESET} --games {TRAIN['games']} --train-steps {TRAIN['train_steps']} --concurrency {TRAIN['concurrency']} --selfplay-workers {TRAIN['selfplay_workers']} --replay-buffer {TRAIN['replay_buffer']} --replay-window {TRAIN['replay_window']} --sims {TRAIN['sims']} --draw-penalty {TRAIN['draw_penalty']} --value-target {TRAIN['value_target']} {resign_arg} {syzygy_arg} --save-every {TRAIN['save_every']} --gate-every 0 --quick-eval-games 0 --lr {TRAIN['lr']} --lr-min {TRAIN['lr_min']} --lr-total-iters {TRAIN['lr_total_iters']} --lr-warmup-iters {TRAIN['lr_warmup_iters']} --grad-clip {TRAIN['grad_clip']} --move-temperature {TRAIN['move_temperature']} --move-temperature-plies {TRAIN['move_temperature_plies']} --checkpoint-dir "$CKPT_DIR" {resume_arg} {reset_opt_arg}


In [ ]:
# 7. Manual gate (optional) — uses gate_games / gate_sims / gate_exploration_moves / gate_openings from TRAIN.
# Edit CHECKPOINT_A / CHECKPOINT_B below (int iteration or 'latest').
# Examples: latest vs 80 (absolute strength); 120 vs 100 (recent progress).
import os
import chess.syzygy
import torch
from engine.config import Config, NetConfig
from engine.encoding import ENCODING_VERSION
from engine.network import ChessNet
from engine.sprt import ALPHA, BETA, ELO0, ELO1, sprt_verdict_label
from engine.train import play_match, _log_gate_metrics, _load_matching_state_dict
from engine.openings import load_default_gate_openings, load_opening_book

if 'TRAIN' not in globals():
    raise RuntimeError('Run cell 6 first (defines TRAIN).')

CHECKPOINT_A = 'latest'   # or e.g. 120
CHECKPOINT_B = 100        # or e.g. 80, 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
cfg.train.draw_penalty = TRAIN['draw_penalty']
cfg.train.syzygy_path = TB_DIR
cfg.train.checkpoint_dir = CKPT_DIR
tablebase = chess.syzygy.open_tablebase(TB_DIR) if os.path.isdir(TB_DIR) else None


def _checkpoint_iteration(checkpoint_ref, state):
    if checkpoint_ref == 'latest':
        if isinstance(state, dict):
            return int(state.get('iteration', -1))
        return -1
    try:
        return int(checkpoint_ref)
    except (TypeError, ValueError):
        return -1


def _load_gate_net(path: str) -> tuple[ChessNet, dict]:
    state = torch.load(path, map_location=DEVICE)
    enc = int(state.get('encoding_version', 1)) if isinstance(state, dict) else 1
    if enc != ENCODING_VERSION:
        raise ValueError(f'{os.path.basename(path)}: encoding {enc} != {ENCODING_VERSION}')
    net_cfg = Config().net
    if isinstance(state, dict) and 'net' in state:
        net_cfg = NetConfig(**state['net'])
    net = ChessNet(net_cfg).to(DEVICE)
    model = state['model'] if isinstance(state, dict) and 'model' in state else state
    _load_matching_state_dict(net, model, label=f'gate load {os.path.basename(path)}', verbose=False)
    net.eval()
    return net, state



def _resolve_gate_openings(spec):
    s = str(spec or 'none').strip()
    if s.lower() in {'', 'none', 'off'}:
        return None
    if s.lower() == 'masters':
        return load_default_gate_openings()
    return load_opening_book(s)


# Resolve Path A
if CHECKPOINT_A == 'latest':
    path_a = os.path.join(CKPT_DIR, 'latest.pt')
    label_a = "Latest"
else:
    path_a = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_A:04d}.pt')
    label_a = f"Iter {CHECKPOINT_A}"

# Resolve Path B
if CHECKPOINT_B == 'latest':
    path_b = os.path.join(CKPT_DIR, 'latest.pt')
    label_b = "Latest"
else:
    path_b = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_B:04d}.pt')
    label_b = f"Iter {CHECKPOINT_B}"

if not os.path.exists(path_a) or not os.path.exists(path_b):
    print(f"Error: Make sure both checkpoints exist!\nPath A: {path_a}\nPath B: {path_b}")
else:
    print(f"Loading Checkpoint A ({label_a}) from {path_a}...")
    net_a, state_a = _load_gate_net(path_a)

    print(f"Loading Checkpoint B ({label_b}) from {path_b}...")
    net_b, state_b = _load_gate_net(path_b)

    gate_openings = _resolve_gate_openings(TRAIN.get('gate_openings', 'masters'))
    book_note = (
        f", book={TRAIN.get('gate_openings', 'masters')} ({len(gate_openings)} lines)"
        if gate_openings else ', book=none'
    )
    print(
        f"\nStarting Match: {label_a} vs {label_b} (SPRT cap {TRAIN['gate_games']} games, "
        f"{TRAIN['gate_sims']} sims, workers={TRAIN.get('gate_workers', TRAIN.get('selfplay_workers', 1))}, "
        f"exploration={TRAIN['gate_exploration_moves']}{book_note})..."
    )

    metrics = play_match(
        net_a, net_b, cfg,
        n_games=TRAIN['gate_games'],
        sims=TRAIN['gate_sims'],
        device=DEVICE,
        exploration_moves=TRAIN['gate_exploration_moves'],
        tablebase=tablebase,
        sprt=True,
        sprt_elo0=ELO0,
        sprt_elo1=ELO1,
        sprt_alpha=ALPHA,
        sprt_beta=BETA,
        workers=TRAIN.get('gate_workers', TRAIN.get('selfplay_workers', 1)),
        concurrency=TRAIN.get('gate_concurrency', 128),
        openings=gate_openings,
    )
    if tablebase is not None:
        tablebase.close()
    winrate = metrics['winrate']
    wins = metrics['wins_as_white'] + metrics['wins_as_black']
    losses = metrics['losses_as_white'] + metrics['losses_as_black']
    draws = metrics['draws_as_white'] + metrics['draws_as_black']

    iter_a = _checkpoint_iteration(CHECKPOINT_A, state_a)
    iter_b = _checkpoint_iteration(CHECKPOINT_B, state_b)
    _log_gate_metrics(CKPT_DIR, iter_a, iter_b, metrics, TRAIN['gate_games'])

    wdl_notation = f"+{wins} ={draws} -{losses}"

    print("\n" + "="*40)
    print(f"MATCH COMPLETED!")
    print(f"{label_a} score vs {label_b}: {winrate:.3f} [{wdl_notation}] ({metrics['games_played']} games)")
    print(f"  As White: Wins: {metrics['wins_as_white']}, Losses: {metrics['losses_as_white']}, Draws: {metrics['draws_as_white']}")
    print(f"  As Black: Wins: {metrics['wins_as_black']}, Losses: {metrics['losses_as_black']}, Draws: {metrics['draws_as_black']}")
    print(f"  Mean Game Length: {metrics['mean_game_len']:.1f} plies")
    print(f"  Terminations: {metrics['terminations']}")
    print(f"  SPRT: {sprt_verdict_label(metrics['sprt_decision'])} (llr={metrics['llr']:.2f}, games={metrics['games_played']})")
    print(f"  Logged to: {os.path.join(CKPT_DIR, 'metrics_gates.csv')}")
    print(f"Result: {label_a} {sprt_verdict_label(metrics['sprt_decision'])} vs {label_b}")
    print("="*40)


In [ ]:
# 8. Plot training progress (run after at least one iteration of cell 6 finishes).
# At 128 games/iter + 800 train steps, one iteration is ~30 min on a T4.
import os
import pandas as pd
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 6 to finish '
          '(several minutes at 128 games on T4), then re-run this cell.')
else:
    df = pd.read_csv(path)
    if df.empty:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        df['iter'] = pd.to_numeric(df['iter'], errors='coerce')
        df = df.dropna(subset=['iter']).copy()
        if df.empty:
            print('metrics.csv has no valid iteration rows yet.')
        else:
            for col in df.columns:
                if col != 'terminations':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            if {'train_steps', 'batch_size', 'samples'}.issubset(df.columns):
                df['reuse_ratio'] = (
                    df['train_steps'] * df['batch_size'] /
                    df['samples'].replace(0, float('nan'))
                )

            # Gate results live in a separate file (logged independently of
            # metrics.csv so a gate crash never loses per-iteration metrics).
            gates_path = os.path.join(CKPT_DIR, 'metrics_gates.csv')
            gates_df = pd.read_csv(gates_path) if os.path.exists(gates_path) else None
            if gates_df is not None and not gates_df.empty:
                gates_df['iter'] = pd.to_numeric(gates_df['iter'], errors='coerce')
                gates_df['winrate'] = pd.to_numeric(gates_df['winrate'], errors='coerce')
                if 'llr' in gates_df.columns:
                    gates_df['llr'] = pd.to_numeric(gates_df['llr'], errors='coerce')
                if 'games_played' in gates_df.columns:
                    gates_df['games_played'] = pd.to_numeric(gates_df['games_played'], errors='coerce')
                gates_df = gates_df.dropna(subset=['iter']).copy()

            # Early-run health checks (first 10-20 iterations of a fresh run).
            df_window = df[df['iter'] <= 20].copy()
            if len(df_window) >= 10:
                entropy_ok = float(df_window['policy_entropy'].iloc[-1]) >= 1.8
                sign_acc_series = df_window['value_sign_acc'].dropna()
                value_ok = (not sign_acc_series.empty) and float(sign_acc_series.iloc[-1]) >= 0.6
                gate_rows = int((gates_df['iter'] <= 20).sum()) if gates_df is not None and not gates_df.empty else 0
                gate_ok = gate_rows >= 1
                reuse_series = df_window['reuse_ratio'].dropna() if 'reuse_ratio' in df_window.columns else pd.Series(dtype=float)
                reuse_ok = (not reuse_series.empty) and float(reuse_series.iloc[-1]) <= 12.0
                print('Early-run checks (iter <= 20):')
                print(f"  policy_entropy >= 1.8: {entropy_ok}")
                print(f"  value_sign_acc >= 0.6: {value_ok}")
                print(f"  gate rows present: {gate_ok} (rows={gate_rows})")
                print(f"  reuse_ratio <= 12x (expect ~6-10): {reuse_ok}")
            else:
                print('Early-run checks need at least 10 completed iterations.')

            fig, ax = plt.subplots(3, 3, figsize=(18, 12))
            ax = ax.flatten()

            def plot_panel(axis, cols, title, ylim=None):
                present = [c for c in cols if c in df.columns]
                if not present:
                    axis.text(0.5, 0.5, 'missing columns', ha='center', va='center')
                    axis.set_title(title)
                    axis.set_xlabel('iteration')
                    return
                df.plot(x='iter', y=present, ax=axis, title=title)
                axis.set_xlabel('iteration')
                if ylim is not None:
                    axis.set_ylim(*ylim)

            plot_panel(ax[0], ['policy_loss', 'value_loss'], 'Losses')
            plot_panel(ax[1], ['policy_entropy', 'grad_norm'], 'Entropy / Gradient norm')
            plot_panel(ax[2], ['value_sign_acc', 'policy_top1_agree'], 'Net target agreement', ylim=(0.0, 1.0))
            plot_panel(ax[3], ['decisive_rate', 'draw_rate', 'max_moves_trunc_rate'], 'Outcome mix', ylim=(0.0, 1.0))
            plot_panel(ax[4], ['mean_game_len'], 'Mean game length')

            if gates_df is None or gates_df.empty:
                ax[5].text(0.5, 0.5, 'no gate rows yet', ha='center', va='center')
                ax[5].set_title('Manual SPRT gate')
                ax[5].set_xlabel('iteration')
            else:
                gates_df.plot(
                    x='iter',
                    y='winrate',
                    marker='o',
                    ax=ax[5],
                    title='Manual SPRT gate winrate',
                    legend=False,
                )
                ax[5].set_xlabel('iteration')
                ax[5].axhline(0.5, color='gray', linestyle='--', linewidth=1)
                ax[5].set_ylim(0.0, 1.0)
                if 'decision' in gates_df.columns:
                    gate_cols = [c for c in [
                        'iter', 'prev_iter', 'games_played',
                        'wins_as_white', 'wins_as_black', 'losses_as_white', 'losses_as_black',
                        'draws_as_white', 'draws_as_black',
                        'winrate', 'wins', 'draws', 'losses', 'mean_game_len',
                        'elo', 'elo_lower', 'elo_upper', 'los', 'llr', 'decision', 'verdict',
                    ] if c in gates_df.columns]
                    print('Recent gates:')
                    print(gates_df[gate_cols].tail(5).to_string(index=False))

            plot_panel(ax[6], ['learning_rate'], 'Learning rate')
            plot_panel(ax[7], ['reuse_ratio'], 'Reuse ratio (train load / fresh samples)')
            plot_panel(ax[8], ['buffer_size'], 'Replay buffer size')

            plt.tight_layout()
            plt.show()

In [ ]:
# 9. Persist checkpoints before the session ends.
# /kaggle/working is wiped when the session stops unless you Save Version / download / Dataset-update.
import os

print('Checkpoint dir:', CKPT_DIR)
print('Contents:')
for name in sorted(os.listdir(CKPT_DIR))[:40]:
    path = os.path.join(CKPT_DIR, name)
    kind = 'dir' if os.path.isdir(path) else 'file'
    print(f'  {kind:4} {name}')
print()
print('Next steps:')
print('  1. Download /kaggle/working/results (or Save Version of this notebook).')
print('  2. Create/update a private Dataset from that folder (include latest.pt + metrics.csv).')
print('  3. Next session: Add Input → that Dataset → re-run cells 1→6.')


## After training

1. Download `latest.pt` from `/kaggle/working/results` (or from your results Dataset).
2. Verify encoding locally:

```bash
python -m engine.inspect_encoding checkpoints/latest.pt
```

3. Start the analysis server:

```bash
# Windows (PowerShell)
$env:IMMORTALITE_ZERO_CHECKPOINT="checkpoints\latest.pt"
python -m uvicorn server.app:app --port 8000
```

4. Open **http://localhost:8000/app/**

Training recipe changes over time: see `TRAINING_CHANGELOG.md` in the repo.
